# LC 84 — Largest Rectangle in Histogram
**Day 45 | Theme: Monotonic Stack | Difficulty: Hard**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Keep a <em>monotonic increasing</em> stack
of indices. When a bar shorter than the top arrives, every taller bar
on the stack is the height of a rectangle that can no longer extend
rightward — pop and compute its maximum width immediately.
</div>

## Official Problem Statement

Given an array of integers `heights` representing the histogram's bar
heights where the width of each bar is `1`, return the area of the
largest rectangle in the histogram.

**Constraints:**
- `1 <= heights.length <= 10^5`
- `0 <= heights[i] <= 10^4`

## What This Is Actually Asking

A rectangle spans a contiguous range of bars and is limited in height
by the shortest bar in that range. We want the largest possible area
(height × width) across all such ranges. Brute force tests every
pair of left/right boundaries — O(n²). The key observation is that
every maximal rectangle has its height equal to some bar in the array;
a monotonic stack finds, for each bar, exactly how far left and right
it can extend as the limiting height in O(n) total work.

## Walk Through an Example by Hand

```
heights = [2, 1, 5, 6, 2, 3]
indices  [ 0, 1, 2, 3, 4, 5]
Append sentinel 0 → heights becomes [2,1,5,6,2,3,0]
```

| i | h  | Stack before | Action                                | area |
|---|----|-------------|---------------------------------------|------|
| 0 | 2  | []          | push 0                                | 0    |
| 1 | 1  | [0]         | 1<2→pop0,h=2,w=1(stack empty→w=i=1); area=2; push1 | 2 |
| 2 | 5  | [1]         | 5>1→push2                             | 2    |
| 3 | 6  | [1,2]       | 6>5→push3                             | 2    |
| 4 | 2  | [1,2,3]     | 2<6→pop3,h=6,w=4-2-1=1,area=6; 2<5→pop2,h=5,w=4-1-1=2,area=10; 2>1→push4 | 10 |
| 5 | 3  | [1,4]       | 3>2→push5                             | 10   |
| 6 | 0  | [1,4,5]     | 0<3→pop5,h=3,w=6-4-1=1,area=3; 0<2→pop4,h=2,w=6-1-1=4,area=8; 0<1→pop1,h=1,w=6(stack empty),area=6; push6 | 10 |

**Result:** `10`  (bar height=5 spanning indices 2-3, width=2)

## The Picture

```
Height
  6 |       █
  5 |     █ █
  4 |     █ █
  3 |     █ █       █
  2 | █   █ █ █     █
  1 | █ █ █ █ █ █   █
      ---------------
      0 1 2 3 4 5

Largest rectangle (area=10): ████████████  ← height 5, width 2
  spans indices 2-3

Stack state (increasing heights, indices stored):
  After i=0: [0]       heights: [2]
  After i=1: [1]       heights: [1]  ← 2 popped, area 2 logged
  After i=3: [1,2,3]   heights: [1,5,6]
  After i=4: [1,4]     heights: [1,2]  ← 6,5 popped, area 6,10
  Sentinel:  []        all remaining bars flushed

Width formula when popping index p at position i:
  stack empty → width = i
  stack not empty → width = i - stack[-1] - 1
```

**Key visual:** The stack is an ascending staircase of heights.
A new shorter bar collapses the staircase, computing areas on the way.

## When To Use This Pattern

- When you need the **largest bounded area** in a histogram-like
  structure, think monotonic increasing stack.
- When a **shorter element limits how far a taller element extends**,
  think stack pop with width calculation.
- When the problem involves "maximal rectangle" in 2-D grids, think
  reduce each row to a histogram + this pattern.
- When appending a **sentinel (0)** simplifies final flush logic,
  think histogram stack problems.
- When brute force is O(n²) or O(n³) area checks, think O(n) stack.

## The Approach

Append a sentinel `0` to `heights` so every bar is guaranteed to be
eventually popped. Maintain a monotonic increasing stack of indices.
When `heights[i]` is less than the bar at the top of the stack, that
top bar cannot extend rightward anymore — pop it, compute its rectangle
area using the current index and the new stack top as boundaries, and
track the running maximum. Continue popping until the stack is empty
or the top bar is shorter, then push `i`.

In [16]:
from typing import List

In [17]:
def test_harness(func):
    """Run test cases for Largest Rectangle in Histogram."""
    cases = [
        # (heights, expected)
        ([2, 1, 5, 6, 2, 3], 10),
        ([2, 4],              4),
        ([1],                 1),
        # Edge: all same height
        ([3, 3, 3, 3],        12),
        # Edge: strictly increasing
        ([1, 2, 3, 4, 5],     9),
        # Edge: strictly decreasing
        ([5, 4, 3, 2, 1],     9),
        # Edge: single zero-height bar
        ([0],                 0),
        # Edge: valley shape
        ([4, 1, 4],           4),
    ]
    passed = 0
    for i, (heights, expected) in enumerate(cases):
        result = func(heights)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"  Case {i}: {status}")
            print(f"    Input:    {heights}")
            print(f"    Expected: {expected}")
            print(f"    Got:      {result}")
    total = len(cases)
    print(f"\n  Summary: {passed}/{total} passed")
    if passed == total:
        print("  All tests PASSED!")

In [18]:
from typing import List
def largestRectangleArea(heights: List[int]) -> int:
    """
    Return the area of the largest rectangle in a histogram.

    Strategy: Monotonic increasing stack of indices.
    - Append sentinel 0 to flush all bars at the end.
    - For each bar: pop taller bars and compute their max
      rectangle width using left boundary = new stack top.
    - Push current index.

    Args:
        heights: Bar heights of histogram (0-indexed, width=1).

    Returns:
        Maximum rectangle area.

    Time:  O(n) — each index pushed/popped at most once.
    Space: O(n) — stack.
    """
    stack = []            # for keeping indexes for a monotonic increasing stack of bars
                          # each element is a list of 2 [height, idx]
                          # the append-ed in the inserted pair is inherited from the index of the last evicted pair
    
    max_area = 0
    for i, h in enumerate(heights):
        cached = i
        while stack and h < stack[-1][1]:
            [p_idx, p_height] = stack.pop()
            cached = p_idx
            max_area = max(max_area , p_height*(i - p_idx))
        stack.append([cached, h])
    while stack:
        pair = stack.pop()
        max_area = max(max_area , pair[1] * (len(heights) - pair[0] ) )
    return max_area
    
print(largestRectangleArea([2,1,5,6,2,3]))  # 10  
print(largestRectangleArea([2,4]))           # 4
print(largestRectangleArea([1]))             # 1
print(largestRectangleArea([6,6,6,6]))       # 24
print(largestRectangleArea([1,2,3,4,5]))     # 9
test_harness(largestRectangleArea)
    


10
4
1
24
9

  Summary: 8/8 passed
  All tests PASSED!


In [ ]:
# Uncomment and run when solution is ready
# test_harness(largest_rectangle_area)

## Complexity

| Approach        | Time   | Space  | Notes                             |
|-----------------|--------|--------|-----------------------------------|
| Brute Force     | O(n²)  | O(1)   | All left/right boundary pairs     |
| Divide & Conquer| O(n log n) | O(n) | Recurse on min-height split     |
| Monotonic Stack | O(n)   | O(n)   | Each index pushed/popped once     |

## Real World Connection

In **Citi** trading systems, this pattern models "maximum contiguous
exposure window" — find the longest sequence of trading days where a
risk metric stays above a floor, maximizing total exposure area. On
**AWS**, storage capacity planning for DynamoDB partitions uses a
histogram of hourly throughput; the largest rectangle corresponds to
the optimal provisioned capacity block that covers demand without
over-provisioning. In **data engineering**, columnar data quality
checks scan schema-level row counts (histogram bars) to find the
widest time window where data completeness exceeds a threshold —
a direct analogue of the largest valid rectangle.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra